In [ ]:
# This program is to compare DMO vs hydro for TNG50-1, later extend it to TNG100

In [1]:
import sys
import ast
import os
sys.path.append(os.path.abspath('..'))

import pandas as pd 
import matplotlib.pyplot as plt
import IPython
import numpy as np 
from dwarf_plotting import plot_lg_virgo, plot_lg_vd, plot_lg_virgo_some
# from Pradyumna_plot import plot_tng_subhalos #This is to plot the subhalos from TNG
import matplotlib
import illustris_python as il
from populating_stars import *
from subhalo_profiles import NFWProfile
from colossus.cosmology import cosmology
from colossus.halo import concentration
from plot_sachi import plot_sachi
from scipy.optimize import fsolve
from scipy import stats
from mpl_toolkits.axes_grid1 import make_axes_locatable
import matplotlib.collections as mc
from matplotlib.legend_handler import HandlerTuple
from testing_errani import get_rot_curve, get_rmxbyrmx0, get_vmxbyvmx0, get_mxbymx0, get_LbyL0, l10rbyrmx0_1by4_spl,l10rbyrmx0_1by2_spl, l10rbyrmx0_1by8_spl, l10rbyrmx0_1by16_spl, l10vbyvmx0_1by2_spl, l10vbyvmx0_1by4_spl, l10vbyvmx0_1by8_spl, l10vbyvmx0_1by16_spl, l10rbyrmx0_1by66_spl, l10rbyrmx0_1by250_spl, l10rbyrmx0_1by1000_spl, l10vbyvmx0_1by66_spl, l10vbyvmx0_1by250_spl, l10vbyvmx0_1by1000_spl
import matplotlib.ticker as ticker
from errani_plus_tng_subhalo import Subhalo
import re

/bigdata/saleslab/psadh003/tng50/dwarf_formation/dwarf_plotting.py:55: RuntimeWarning: invalid value encountered in sqrt
  rh_proj *= np.sqrt(1.-ell)


A new version of galpy (1.10.1) is available, please upgrade using pip/conda/... to get the latest features and bug fixes!


In [ ]:
cosmology.setCosmology('planck18')

font = {'family' : 'DejaVu Sans',
        'weight' : 'normal',
        'size' : 14}

plt.figure()
plt.close()
matplotlib.rc('font', **font)

In [ ]:
label_font = 20

In [ ]:
import illustris_python as il

def convert_to_float(value):
    try:
        if isinstance(value, float) or isinstance(value, int):
            return value

        if value == '[inf, inf, inf]' or value == '[inf inf inf]':
            return np.array([np.inf, np.inf, np.inf])

        if value == '[inf]':
            return np.inf

        if value == '[-inf]':
            return -np.inf

        

        blah = ast.literal_eval(value)
        if isinstance(blah, list):
            if len(blah) == 1:
                blah2 = float(blah[0])   
            elif len(blah) == 3:
                blah2 = np.array([float(blah[0]), float(blah[1]), float(blah[2])])
        else:
            blah2 = float(blah)        
        return blah2
    except Exception as e:
        try:
            value_fixed = re.sub(r'(?<=\d)\s+(?=-?\d)', ', ', value)  # Handle spaces between numbers
            value_fixed = re.sub(r'(?<=\d)\s+(?=\d)', ', ', value_fixed)
            value_fixed = re.sub(r'(?<=\.)\s+(?=-\d)', ', ', value_fixed)
            value_fixed = re.sub(r'(?<=\.)\s+(?=\d)', ', ', value_fixed)
            # Now use ast.literal_eval
            # print(value_fixed)
            result = ast.literal_eval(value_fixed)
            return result
        except Exception as e:
            print(value)
            return value
    # except ValueError:
    #     return value


def get_med_values(arr, cuts):
    '''
    This is going to take one big array of length cuts*whatever, breaks it into cuts parts and returns medians of the corresponding elements
    '''
    return np.median(np.reshape(arr, (cuts, -1)), axis = 0)

def get_quantiles(arr, cuts):
    '''
    This is going to take one big array of length cuts*whatever, breaks it into cuts parts and returns medians of the corresponding elements
    '''
    return np.quantile(np.reshape(arr, (cuts, -1)), axis = 0, q = [0.05, 0.95])


basePath = '/rhome/psadh003/bigdata/L35n2160TNG_fixed/output'
outpath  = '/rhome/psadh003/bigdata/tng50/output_files/'
plotpath = '/bigdata/saleslab/psadh003/tng50/output_plots/'
misc_path = '/bigdata/saleslab/psadh003/misc_files/'
ferrarese_data_path = '/bigdata/saleslab/psadh003/misc_files/Ferrarese_virgo_smf.csv'
venhola_data_path = '/bigdata/saleslab/psadh003/misc_files/Venhola_fornax_smf.csv'

'''
This part is to plot the NGVS data of virgo core set of satellites
'''
fdata = pd.read_csv(ferrarese_data_path, delimiter = ',')
fmstar = fdata['mstar']
fngal = fdata['ngal']
fngal_cum = np.cumsum(fngal)

'''
This part is to plot the Fornax mass function (everything inside virial radius)
'''
vdata = pd.read_csv(venhola_data_path, delimiter = ',')
vmstar = vdata['mstar']
vngal = vdata['ngal']
vngal_cum = np.cumsum(vngal)



fof_no = 210
fof_str = 'fof210'

this_fof0 = il.groupcat.loadSingle(basePath, 99, haloID = 0)
central_sfid_99_0 = this_fof0['GroupFirstSub']
rvir_fof0 = this_fof0['Group_R_Crit200']/0.6744
mvir_fof0 = this_fof0['Group_M_Crit200']*1e10/0.6744

this_fof1 = il.groupcat.loadSingle(basePath, 99, haloID = 1)
central_sfid_99_1 = this_fof1['GroupFirstSub']
rvir_fof1 = this_fof1['Group_R_Crit200']/0.6744
mvir_fof1 = this_fof1['Group_M_Crit200']*1e10/0.6744

this_fof2 = il.groupcat.loadSingle(basePath, 99, haloID = 2)
central_sfid_99_2 = this_fof2['GroupFirstSub']
rvir_fof2 = this_fof2['Group_R_Crit200']/0.6744
mvir_fof2 = this_fof2['Group_M_Crit200']*1e10/0.6744

this_fof_plotppath = '/bigdata/saleslab/psadh003/tng50/final_plots/' + fof_str + '/'
if not os.path.exists(this_fof_plotppath): #If the directory does not exist, then just create it!
    os.makedirs(this_fof_plotppath)



'''
In the following lines, 
(i) ps stands for power law values of surviving subhalos
(ii) pm stands for power law values of merged subhalos
(iii) cs stands for cutoff values of surviving subhalos
(iv) cm stands for cutoff values of merged subhalos
'''

mcutoff = 1e2


if True: #This is a section for getting power law values of survived subhalos
    pdfs = pd.read_csv(outpath + fof_str + '_surviving_evolved_everything.csv', delimiter = ',', low_memory=False)
    pdfs = pdfs.applymap(convert_to_float)


    # subset1 = pdfs[(pdfs['dist_f_ar']<rvir_fof0) & (pdfs['fof'] == 0)]
    # subset2 = pdfs[(pdfs['dist_f_ar']<rvir_fof1) & (pdfs['fof'] == 1)]
    # subset3 = pdfs[(pdfs['dist_f_ar']<rvir_fof2) & (pdfs['fof'] == 2)]

    # pdfs = pd.concat([subset1, subset2, subset3], axis=0)
    pdfs1 = pdfs
    # if porc == 'p':
        # print(len(pdfs[pdfs['mstar_f_ar_tng'].values>5e6]))
    columns_to_max = ['mstar_max_ar', 'mstar_max_pl_ar']
    max_values = pdfs[columns_to_max].max(axis=1)
    pdfs = pdfs[(max_values > mcutoff)]

    columns_to_max = ['mstar_f_ar_tng', 'mstar_f_pl_ar']
    max_values = pdfs[columns_to_max].max(axis=1)
    cutoff = 10

    pdfs = pdfs[(max_values > cutoff)]
    # pdfs = pdfs[pdfs[]]


    psdist_f_ar = pdfs['dist_f_ar'].values
    pspos_f_ar = np.stack(np.array(pdfs['pos_f_ar'].values))
    psvel_f_ar = np.stack(np.array(pdfs['vel_f_ar'].values))

    pspos_if_ar = np.stack(np.array(pdfs['pos_if_ar'].values))
    psdist_if_ar = np.linalg.norm(pspos_if_ar, axis = 1)
    psvel_if_ar = np.stack(np.array(pdfs['vel_if_ar'].values))
    # print(pspos_f_ar)
    # print(pspos_f_ar.shape)
    psxydist_f_ar = np.sqrt(np.sum(np.square(pspos_f_ar[:, :2]), axis = 1))
    psyzdist_f_ar = np.sqrt(np.sum(np.square(pspos_f_ar[:, 1:]), axis = 1))
    psxzdist_f_ar = np.sqrt(np.sum(np.square(pspos_f_ar[:, [0, 2]]), axis = 1))


    psvd_f_ar_tng = pdfs['vd_f_ar_tng'].values
    psrh_f_ar_tng = pdfs['rh_f_ar_tng'].values
    psmstar_f_ar_tng = pdfs['mstar_f_ar_tng'].values
    psmmx_f_ar_tng = pdfs['mmx_f_ar_tng'].values
    psrmx_f_ar_tng = pdfs['rmx_f_ar_tng'].values
    psvmx_f_ar_tng = pdfs['vmx_f_ar_tng'].values

    pstinf_ar = pdfs['tinf_ar'].values
    pstorb_ar = pdfs['torb_ar'].values
    psrapo_ar = pdfs['rapo_ar'].values
    psrperi_ar = pdfs['rperi_ar'].values

    psvd_f_ar = pdfs['vd_f_ar'].values
    psrh_f_ar = pdfs['rh_f_ar'].values
    psmstar_f_ar = pdfs['mstar_f_ar'].values

    psvd_max_ar = pdfs['vd_max_ar'].values
    psrh_max_ar = pdfs['rh_max_ar'].values
    psmstar_max_ar = pdfs['mstar_max_ar'].values

    pssnap_if_ar = pdfs['snap_if_ar'].values
    pssf_id_if_ar = pdfs['sfid_if_ar'].values

    psmmx_f_ar = pdfs['mmx_f_ar'].values
    psrmx_f_ar = pdfs['rmx_f_ar'].values
    psvmx_f_ar = pdfs['vmx_f_ar'].values

    psmmx_if_ar = pdfs['mmx_if_ar'].values
    psrmx_if_ar = pdfs['rmx_if_ar'].values
    psvmx_if_ar = pdfs['vmx_if_ar'].values

    psvd_f_pl_ar = pdfs['vd_f_pl_ar'].values
    psrh_f_pl_ar = pdfs['rh_f_pl_ar'].values
    psmstar_f_pl_ar = pdfs['mstar_f_pl_ar'].values
    psvd_f_co_ar = pdfs['vd_f_co_ar'].values
    psrh_f_co_ar = pdfs['rh_f_co_ar'].values
    psmstar_f_co_ar = pdfs['mstar_f_co_ar'].values
    psrh_max_pl_ar = pdfs['rh_max_pl_ar'].values
    psrh_max_co_ar = pdfs['rh_max_co_ar'].values
    psmstar_max_pl_ar = pdfs['mstar_max_pl_ar'].values
    psmstar_max_co_ar = pdfs['mstar_max_co_ar'].values
    psfof = pdfs['fof'].values
    psmstar_if_ar = pdfs['mstar_if_ar'].values
    psvmax_if_ar = pdfs['vmax_if_ar'].values
    
    psmbp_pos_f_ar = pdfs['mbp_pos_f_ar'].values
    psmbp_vel_f_ar = pdfs['mbp_vel_f_ar'].values
    psmbp_dist_f_ar = pdfs['mbp_dist_f_ar'].values
    psgr_sat = pdfs['grp_status'].values

    pspot_f_ar = pdfs['pot_f_ar'].values
    pspot_if_ar = pdfs['pot_if_ar'].values
    psmpeak_ar = pdfs['mpeak_ar'].values
    psmdm_f_ar = pdfs['mdm_f_ar'].values
    

    

    psmstar_all = np.zeros(len(psmmx_f_ar))
    psrh_all = np.zeros(len(psmmx_f_ar))
    psvd_all = np.zeros(len(psmmx_f_ar))
    res_ixs = np.where(psmstar_max_ar >= 5e6)[0] #These are the resolved indices
    unres_ixs = np.where(psmstar_max_ar < 5e6)[0] #These are the unresolved indices

    psmstar_max_all = np.zeros(len(psmmx_f_ar))
    psrh_max_all = np.zeros(len(psmmx_f_ar))

    psmstar_max_all[res_ixs] = psmstar_max_ar[res_ixs]
    psmstar_max_all[unres_ixs] = psmstar_max_pl_ar[unres_ixs]

    psrh_max_all[res_ixs] = psrh_max_ar[res_ixs]
    psrh_max_all[unres_ixs] = psrh_max_pl_ar[unres_ixs]

    psmstar_all[res_ixs] = psmstar_f_ar[res_ixs]
    psmstar_all[unres_ixs] = psmstar_f_pl_ar[unres_ixs]
    psrh_all[res_ixs] = psrh_f_ar[res_ixs]
    psrh_all[unres_ixs] = psrh_f_pl_ar[unres_ixs]
    psvd_all[res_ixs] = psvd_f_ar[res_ixs]
    psvd_all[unres_ixs] = psvd_f_pl_ar[unres_ixs]


    pssigma_all = 4.83 +21.57 -2.5 * np.log10(psmstar_all / (np.pi * (psrh_all*1e3) ** 2)) #This will be in mag/arcsec^2





if True: #This is a section for getting power law values of merged subhalos
    pdfm = pd.read_csv(outpath + fof_str + '_merged_evolved_wmbp_everything.csv', delimiter = ',')
    # print(pdfm.head(20))
    # print(f'Length before is: {len(pdfm)}')
    # pdfm = pdfm[pdfm['dist_f_ar'] != '']
    # print(f'Length after is: {len(pdfm)}')

    # pdfm = pdfm.dropna(subset = ['dist_f_ar'])
    # print(pdfm['pos_f_ar'])
    pdfm = pdfm.applymap(convert_to_float)
    # pdfm = pdfm[pdfm['dist_f_ar']<rvir_fof]

    # subset1 = pdfm[(pdfm['dist_f_ar']<rvir_fof0) & (pdfm['fof'] == 0)]
    # subset2 = pdfm[(pdfm['dist_f_ar']<rvir_fof1) & (pdfm['fof'] == 1)]
    # subset3 = pdfm[(pdfm['dist_f_ar']<rvir_fof2) & (pdfm['fof'] == 2)]

    # pdfm = pd.concat([subset1, subset2, subset3], axis=0)
    pdfm1 = pdfm 

    columns_to_max = ['mstar_max_ar', 'mstar_max_pl_ar']
    max_values = pdfm[columns_to_max].max(axis=1)
    pdfm = pdfm[(max_values > mcutoff)]

    columns_to_max = ['mstar_f_ar', 'mstar_f_pl_ar']

    # Compute the maximum along the specified axis (axis=1 for row-wise)
    max_values = pdfm[columns_to_max].max(axis=1)

    # Sample cutoff value
    cutoff = 10

    # print(len(pdfm))
    pdfm = pdfm[max_values > cutoff]
    # print(len(pdfm))

    pmdist_f_ar = pdfm['dist_f_ar'].values
    # print(pmpos_ar)
    # print(pdfm['pos_f_ar'].values)
    pmpos_list = pdfm['pos_f_ar'].values
    pmvel_list = pdfm['vel_f_ar'].values


    # pmpos_list = pdfm['pos_f_ar'].values
    # pmvel_list = pdfm['vel_f_ar'].values

    
    for idx, arr in enumerate(pmpos_list):
        if np.array(arr).size != 3:  # Check if the array is empty
            pmpos_list[idx] = [np.inf, np.inf, np.inf]
            pmvel_list[idx] = [np.inf, np.inf, np.inf]
            pmdist_f_ar[idx] = np.inf
    pmpos_ar = np.stack(np.array(pmpos_list))
    pmvel_f_ar = np.stack(np.array(pmvel_list))

    pmpos_if_ar = np.stack(np.array(pdfm['pos_if_ar'].values))
    pmdist_if_ar = np.linalg.norm(pmpos_if_ar, axis = 1)
    pmvel_if_ar = np.stack(np.array(pdfm['vel_if_ar'].values))
    # pmpos_ar
    # print(pmpos_ar)
    pmxydist_f_ar = np.sqrt(np.sum(np.square(pmpos_ar[:, :2]), axis = 1))
    pmyzdist_f_ar = np.sqrt(np.sum(np.square(pmpos_ar[:, 1:]), axis = 1))
    pmxzdist_f_ar = np.sqrt(np.sum(np.square(pmpos_ar[:, [0, 2]]), axis = 1))
    pmmbpid_ar = pdfm['mbpid_ar'].values

    pmtinf_ar = pdfm['tinf_ar'].values
    pmtorb_ar = pdfm['torb_ar'].values
    pmrapo_ar = pdfm['rapo_ar'].values
    pmrperi_ar = pdfm['rperi_ar'].values

    pmvd_f_ar = pdfm['vd_f_ar'].values
    pmrh_f_ar = pdfm['rh_f_ar'].values
    pmmstar_f_ar = pdfm['mstar_f_ar'].values

    pmvd_max_ar = pdfm['vd_max_ar'].values
    pmrh_max_ar = pdfm['rh_max_ar'].values
    pmmstar_max_ar = pdfm['mstar_max_ar'].values

    pmsnap_if_ar = pdfm['snap_if_ar'].values
    pmsfid_if_ar = pdfm['sfid_if_ar'].values

    pmmmx_f_ar = pdfm['mmx_f_ar'].values
    pmrmx_f_ar = pdfm['rmx_f_ar'].values
    pmvmx_f_ar = pdfm['vmx_f_ar'].values

    pmmmx_if_ar = pdfm['mmx_if_ar'].values
    pmrmx_if_ar = pdfm['rmx_if_ar'].values
    pmvmx_if_ar = pdfm['vmx_if_ar'].values

    pmvd_f_pl_ar = pdfm['vd_f_pl_ar'].values
    pmrh_f_pl_ar = pdfm['rh_f_pl_ar'].values
    pmmstar_f_pl_ar = pdfm['mstar_f_pl_ar'].values
    pmvd_f_co_ar = pdfm['vd_f_co_ar'].values
    pmrh_f_co_ar = pdfm['rh_f_co_ar'].values
    pmmstar_f_co_ar = pdfm['mstar_f_co_ar'].values
    pmrh_max_pl_ar = pdfm['rh_max_pl_ar'].values
    pmrh_max_co_ar = pdfm['rh_max_co_ar'].values
    pmmstar_max_pl_ar = pdfm['mstar_max_pl_ar'].values
    pmmstar_max_co_ar = pdfm['mstar_max_co_ar'].values
    pmpot_f_ar = pdfm['pot_f_ar'].values
    pmpot_if_ar = pdfm['pot_if_ar'].values
    pmmpeak_ar = pdfm['mpeak_ar'].values


    
    pmfof = pdfm['fof'].values

    pmgp_pos_f_ar = pdfm['gp_pos_f_ar'].values
    pmgp_vel_f_ar = pdfm['gp_vel_f_ar'].values
    pmgp_dist_f_ar = pdfm['gp_dist_f_ar'].values

    pmgr_sat = pdfm['grp_status'].values
    

    pmmstar_max_all = np.zeros(len(pmmmx_f_ar))
    pmrh_max_all = np.zeros(len(pmmmx_f_ar))

    pmmstar_all = np.zeros(len(pmmmx_f_ar))
    pmrh_all = np.zeros(len(pmmmx_f_ar))
    pmvd_all = np.zeros(len(pmmmx_f_ar))
    res_ixs = np.where(pmmstar_max_ar >= 5e6)[0] #These are the resolved indices
    unres_ixs = np.where(pmmstar_max_ar < 5e6)[0] #These are the unresolved indices
    pmmstar_all[res_ixs] = pmmstar_f_ar[res_ixs]
    pmmstar_all[unres_ixs] = pmmstar_f_pl_ar[unres_ixs]

    pmmstar_max_all[res_ixs] = pmmstar_max_ar[res_ixs]
    pmmstar_max_all[unres_ixs] = pmmstar_max_pl_ar[unres_ixs]


    pmrh_max_all[res_ixs] = pmrh_max_ar[res_ixs]
    pmrh_max_all[unres_ixs] = pmrh_max_pl_ar[unres_ixs]



    pmrh_all[res_ixs] = pmrh_f_ar[res_ixs]
    pmrh_all[unres_ixs] = pmrh_f_pl_ar[unres_ixs]
    pmvd_all[res_ixs] = pmvd_f_ar[res_ixs]
    pmvd_all[unres_ixs] = pmvd_f_pl_ar[unres_ixs]

    pmsigma_all = 4.83 +21.57 -2.5 * np.log10(pmmstar_all / (np.pi * (pmrh_all*1e3) ** 2)) #This will be in mag/arcsec^2




#Once we have the power law data for surviving and merged subhalos, we should obtain their respective stellar masses etc checking if they are resolved




In [ ]:
filepath = '/rhome/psadh003/bigdata/tng50/tng_files/'
outpath  = '/rhome/psadh003/bigdata/tng50/output_files/'
baseUrl = 'https://www.tng-project.org/api/TNG50-1/'
headers = {"api-key":"894f4df036abe0cb9d83561e4b1efcf1"}
basePath = '/rhome/psadh003/bigdata/L35n2160TNG_fixed/output'
fof_path = '/bigdata/saleslab/psadh003/tng50/fof_partdata/'
misc_path = '/bigdata/saleslab/psadh003/misc_files/'\

G = 4.3e-6
h = 0.6744
this_fof = il.groupcat.loadSingle(basePath, 99, haloID = 0)

central_sfid_99 = this_fof['GroupFirstSub']
cen = il.groupcat.loadSingle(basePath, 99, subhaloID = central_sfid_99)
m200 = this_fof['Group_M_Crit200'] * 1e10 / h
r200 = this_fof['Group_R_Crit200'] / h
v200 = np.sqrt(G * m200 / r200)
cen['SubhaloVmax']

In [ ]:

this_fof = il.groupcat.loadSingle(basePath, 99, haloID = 1)

central_sfid_991 = this_fof['GroupFirstSub']
cen1 = il.groupcat.loadSingle(basePath, 99, subhaloID = central_sfid_991)
m2001 = this_fof['Group_M_Crit200'] * 1e10 / h
r2001 = this_fof['Group_R_Crit200'] / h
v2001 = np.sqrt(G * m200 / r200)
cen1['SubhaloVmax']

In [ ]:
this_fof = il.groupcat.loadSingle(basePath, 99, haloID = 2)

central_sfid_992 = this_fof['GroupFirstSub']
cen2 = il.groupcat.loadSingle(basePath, 99, subhaloID = central_sfid_992)
m2002 = this_fof['Group_M_Crit200'] * 1e10 / h
r2002 = this_fof['Group_R_Crit200'] / h
v2002 = np.sqrt(G * m200 / r200)
cen2['SubhaloVmax']

In [ ]:
pske_if_ar = 0.5*np.linalg.norm(psvel_if_ar, axis = 1)**2
pske_f_ar = 0.5*np.linalg.norm(psvel_f_ar, axis = 1)**2
pmke_f_ar = 0.5*np.linalg.norm(pmvel_f_ar, axis = 1)**2
pmke_if_ar = 0.5*np.linalg.norm(pmvel_if_ar, axis = 1)**2
pmte_f_ar = pmpot_f_ar + pmke_f_ar
pmte_if_ar = pmpot_if_ar + pmke_if_ar
pste_f_ar = pspot_f_ar + pske_f_ar
pste_if_ar = pspot_if_ar + pske_if_ar


In [ ]:
# We will now have to extract the data from TNG50-1
t50dbasePath = '/rhome/psadh003/bigdata/TNG50-1-Dark/output'
t50dfilepath = '/rhome/psadh003/bigdata/tng50dark/tng_files/'
fofno = 0
tng50ddf = pd.read_csv(t50dfilepath + 'fofno_' + str(fofno) + '.csv', delimiter = ',')
posx0d_ar = tng50ddf['posx_ar'].values
posy0d_ar = tng50ddf['posy_ar'].values
posz0d_ar = tng50ddf['posz_ar'].values
this_fof = il.groupcat.loadSingle(t50dbasePath, 99, haloID = fofno)
fof0d_pos = this_fof['GroupPos']/h 
fof0d_rvir = this_fof['Group_R_Crit200']/h

dist0d_ar = np.sqrt((posx0d_ar - fof0d_pos[0])**2 + (posy0d_ar - fof0d_pos[1])**2 + (posz0d_ar - fof0d_pos[2])**2)
vmax0d_ar = tng50ddf['vmax_ar'].values
len0d_ar = tng50ddf['len_ar'].values


fofno = 1
tng50ddf = pd.read_csv(t50dfilepath + 'fofno_' + str(fofno) + '.csv', delimiter = ',')
posx1d_ar = tng50ddf['posx_ar'].values
posy1d_ar = tng50ddf['posy_ar'].values
posz1d_ar = tng50ddf['posz_ar'].values
this_fof = il.groupcat.loadSingle(t50dbasePath, 99, haloID = fofno)
fof1d_pos = this_fof['GroupPos']/h
fof1d_rvir = this_fof['Group_R_Crit200']/h

dist1d_ar = np.sqrt((posx1d_ar - fof1d_pos[0])**2 + (posy1d_ar - fof1d_pos[1])**2 + (posz1d_ar - fof1d_pos[2])**2)
vmax1d_ar = tng50ddf['vmax_ar'].values
len1d_ar = tng50ddf['len_ar'].values

fofno = 2
tng50ddf = pd.read_csv(t50dfilepath + 'fofno_' + str(fofno) + '.csv', delimiter = ',')
posx2d_ar = tng50ddf['posx_ar'].values
posy2d_ar = tng50ddf['posy_ar'].values
posz2d_ar = tng50ddf['posz_ar'].values
this_fof = il.groupcat.loadSingle(t50dbasePath, 99, haloID = fofno)
fof2d_pos = this_fof['GroupPos']/h
fof2d_rvir = this_fof['Group_R_Crit200']/h

dist2d_ar = np.sqrt((posx2d_ar - fof2d_pos[0])**2 + (posy2d_ar - fof2d_pos[1])**2 + (posz2d_ar - fof2d_pos[2])**2)
vmax2d_ar = tng50ddf['vmax_ar'].values
len2d_ar = tng50ddf['len_ar'].values





In [ ]:
# We will now have to extract the data from TNG100-1
t100dbasePath = '/rhome/psadh003/bigdata/TNG100-1-Dark/output'
t100dfilepath = '/rhome/psadh003/bigdata/tng100dark/tng_files/'


fofno = 0
tng100ddf = pd.read_csv(t100dfilepath + 'fofno_' + str(fofno) + '.csv', delimiter = ',')
posxm0d1_ar = tng100ddf['posx_ar'].values
posym0d1_ar = tng100ddf['posy_ar'].values
poszm0d1_ar = tng100ddf['posz_ar'].values
this_fof = il.groupcat.loadSingle(t100dbasePath, 99, haloID = fofno)
fofm0d1_pos = this_fof['GroupPos']/h
fofm0d1_rvir = this_fof['Group_R_Crit200']/h

distm0d1_ar = np.sqrt((posxm0d1_ar - fofm0d1_pos[0])**2 + (posym0d1_ar - fofm0d1_pos[1])**2 + (poszm0d1_ar - fofm0d1_pos[2])**2)
vmaxm0d1_ar = tng100ddf['vmax_ar'].values
lenm0d1_ar = tng100ddf['len_ar'].values




fofno = 12
tng100ddf = pd.read_csv(t100dfilepath + 'fofno_' + str(fofno) + '.csv', delimiter = ',')
posx0d1_ar = tng100ddf['posx_ar'].values
posy0d1_ar = tng100ddf['posy_ar'].values
posz0d1_ar = tng100ddf['posz_ar'].values
this_fof = il.groupcat.loadSingle(t100dbasePath, 99, haloID = fofno)
fof0d1_pos = this_fof['GroupPos']/h
fof0d1_rvir = this_fof['Group_R_Crit200']/h

dist0d1_ar = np.sqrt((posx0d1_ar - fof0d1_pos[0])**2 + (posy0d1_ar - fof0d1_pos[1])**2 + (posz0d1_ar - fof0d1_pos[2])**2)
vmax0d1_ar = tng100ddf['vmax_ar'].values
len0d1_ar = tng100ddf['len_ar'].values


fofno = 13
tng100ddf = pd.read_csv(t100dfilepath + 'fofno_' + str(fofno) + '.csv', delimiter = ',')
posx1d1_ar = tng100ddf['posx_ar'].values
posy1d1_ar = tng100ddf['posy_ar'].values
posz1d1_ar = tng100ddf['posz_ar'].values
this_fof = il.groupcat.loadSingle(t100dbasePath, 99, haloID = fofno)
fof1d1_pos = this_fof['GroupPos']/h
fof1d1_rvir = this_fof['Group_R_Crit200']/h

dist1d1_ar = np.sqrt((posx1d1_ar - fof1d1_pos[0])**2 + (posy1d1_ar - fof1d1_pos[1])**2 + (posz1d1_ar - fof1d1_pos[2])**2)
vmax1d1_ar = tng100ddf['vmax_ar'].values
len1d1_ar = tng100ddf['len_ar'].values

fofno = 14
tng100ddf = pd.read_csv(t100dfilepath + 'fofno_' + str(fofno) + '.csv', delimiter = ',')
posx2d1_ar = tng100ddf['posx_ar'].values
posy2d1_ar = tng100ddf['posy_ar'].values
posz2d1_ar = tng100ddf['posz_ar'].values
this_fof = il.groupcat.loadSingle(t100dbasePath, 99, haloID = fofno)
fof2d1_pos = this_fof['GroupPos']/h
fof2d1_rvir = this_fof['Group_R_Crit200']/h

dist2d1_ar = np.sqrt((posx2d1_ar - fof2d1_pos[0])**2 + (posy2d1_ar - fof2d1_pos[1])**2 + (posz2d1_ar - fof2d1_pos[2])**2)
vmax2d1_ar = tng100ddf['vmax_ar'].values
len2d1_ar = tng100ddf['len_ar'].values

In [ ]:
print('This is to compare the FoF0 of TNG50-1 with the DMO counterpart')

# ph_dm_particles_r[33] # This is where the r/r200 for Phoenix reaches 0.4


print('This is to make a version where we have all three FoFs and r_apo < 2e3 kpc and everything is normalized to r/r200 = 0.4')
h = 0.6774
this_fof = il.groupcat.loadSingle(basePath, 99, haloID = 0)

central_sfid_99 = this_fof['GroupFirstSub']
cen = il.groupcat.loadSingle(basePath, 99, subhaloID = central_sfid_99)
m200 = this_fof['Group_M_Crit200'] * 1e10 / h
r200 = this_fof['Group_R_Crit200'] / h
v200 = np.sqrt(G * m200 / r200)

ph_dm_particles_df = pd.read_csv(misc_path + 'phoenix_dm_particles.csv', header = None)
ph_dm_particles_r = np.array(ph_dm_particles_df[0])
ph_dm_particles_n = np.array(ph_dm_particles_df[1])

ph_max_vmax_at_infall_df = pd.read_csv(misc_path + 'phoenix_max_vmax_at_infall.csv', header = None)
ph_max_vmax_at_infall_r = np.array(ph_max_vmax_at_infall_df[0])
ph_max_vmax_at_infall_n = np.array(ph_max_vmax_at_infall_df[1])

ph_max_vmax_z0_df = pd.read_csv(misc_path + 'phoenix_max_vmax_z0.csv', header = None)
ph_max_vmax_z0_r = np.array(ph_max_vmax_z0_df[0])
ph_max_vmax_z0_n = np.array(ph_max_vmax_z0_df[1])


# print(ph_dm_particles_r)


# print('This is normal')
def plot_radial_density_dist_3panel(fofno = 0):
    '''
    This is to plot the radial density of subhalos in 3 panels for FoF0 in three mass ranges
    '''
    fig, axs = plt.subplots(1, 1, figsize = (6, 6))
    
    if fofno == 0:
        Ndm_ar =[721148, 778322, 839804, 905710, 976898, 1054137, 1137501, 1227676, 1325214, 1430877, 1545758, 1669730, 1805234, 1953330, 2114837, 2291374, 2484727, 2697632, 2929798, 3184435, 3461447, 3763454, 4093457, 4451911, 4839639, 5261067, 5717234, 6218030, 6759960, 7335930, 7958128, 8628601, 9351223, 10124871, 10954252, 11843815, 12803150, 13852694, 14955612, 16130128, 17390800, 18724037, 20147057, 21672913, 23303629, 25059909, 26945144, 28965678, 31153056, 33453105, 35883715, 38490363, 41284274, 44311781, 47556207, 50917974, 54416144, 58216056, 62174887, 66291888, 70515423, 74848372, 79404305, 84157527, 89209147, 94471028, 99857920, 105317116, 111028328, 116939549, 123289439, 130030026, 137275327, 145069112, 153548719, 161907244, 169905804, 177886839, 185830162, 194592334, 203383663, 212695914, 222350306, 232159462, 242040951, 251514511, 260202731, 268769952, 278511000, 289919552, 300273807, 309815832, 319892274, 331241231, 342176869, 352467656, 365061316, 375410309, 383675808, 390894527]
        Mstar_ar = [1215764100000.0, 1254275500000.0, 1293225400000.0, 1332839300000.0, 1373199400000.0, 1414346300000.0, 1456465800000.0, 1499649100000.0, 1543340700000.0, 1587188200000.0, 1631277400000.0, 1676285400000.0, 1723836500000.0, 1770465500000.0, 1818711300000.0, 1868407000000.0, 1921189400000.0, 1975527500000.0, 2031582000000.0, 2091765200000.0, 2152079200000.0, 2214667500000.0, 2279144400000.0, 2346879800000.0, 2415670000000.0, 2483960200000.0, 2560993300000.0, 2639518500000.0, 2717090000000.0, 2789425800000.0, 2865190000000.0, 2938990200000.0, 3014576300000.0, 3088882300000.0, 3166544000000.0, 3234522200000.0, 3303603000000.0, 3403499700000.0, 3475743700000.0, 3541706200000.0, 3611283200000.0, 3671716800000.0, 3740037800000.0, 3803736700000.0, 3863171600000.0, 3924207300000.0, 3984920700000.0, 4047178600000.0, 4116152300000.0, 4179430200000.0, 4235346300000.0, 4302655000000.0, 4361331100000.0, 4432361700000.0, 4508296400000.0, 4587731300000.0, 4649099700000.0, 4724291500000.0, 4775264400000.0, 4826232000000.0, 4874225000000.0, 4915582000000.0, 4958145500000.0, 4994681500000.0, 5043397300000.0, 5096712000000.0, 5136737400000.0, 5176501500000.0, 5222220400000.0, 5250391500000.0, 5282301000000.0, 5315951000000.0, 5351944000000.0, 5401151000000.0, 5537075300000.0, 5626221000000.0, 5672295500000.0, 5750817000000.0, 5774166000000.0, 5900201400000.0, 5929926000000.0, 5960066700000.0, 5986782300000.0, 6014626000000.0, 6088623000000.0, 6145640500000.0, 6163691000000.0, 6178112000000.0, 6282066000000.0, 6651228000000.0, 6749657000000.0, 6765084700000.0, 6809034700000.0, 6949706000000.0, 6986958000000.0, 7002995600000.0, 7282621400000.0, 7327393000000.0, 7357744000000.0, 7428319000000.0]
    elif fofno == 1:
        # elif fofno == 1:
        Ndm_ar =  [786023, 841839, 902134, 966380, 1035837, 1110113, 1189602, 1274186, 1364366, 1460423, 1562817, 1672316, 1788483, 1913425, 2046885, 2190148, 2343775, 2511901, 2690528, 2878121, 3078467, 3294791, 3528395, 3780942, 4053436, 4350232, 4685298, 5050150, 5416132, 5796879, 6199598, 6623522, 7072307, 7546453, 8050591, 8585909, 9153650, 9756524, 10397233, 11074877, 11795747, 12560524, 13376455, 14255740, 15217979, 16289952, 17349673, 18430107, 19555220, 20756630, 22010268, 23314416, 24673283, 26107659, 27631468, 29312494, 31108073, 32886889, 34739969, 36734573, 38844189, 41049934, 43382025, 45812785, 48395953, 51173751, 54141628, 57270029, 60739449, 64360174, 68074798, 71803090, 75817335, 80041042, 84373691, 89111485, 94087743, 99601105, 104804935, 109994769, 115260983, 121202017, 127487242, 133053679, 138547841, 144456102, 151341815, 158789550, 164815392, 169912646, 174986366, 179518933, 183635753, 187442091, 191210434, 194512610, 197453028, 199783883, 201489573, 202770641]
        Mstar_ar = [735365200000.0, 752437950000.0, 769758000000.0, 786811060000.0, 803612500000.0, 819888000000.0, 835624570000.0, 850837800000.0, 865607700000.0, 879804150000.0, 893492400000.0, 906636800000.0, 919314960000.0, 931422670000.0, 942965800000.0, 953885000000.0, 964442060000.0, 978516050000.0, 990955000000.0, 999915700000.0, 1008391160000.0, 1016919560000.0, 1025774060000.0, 1036594640000.0, 1047105040000.0, 1059257000000.0, 1096440500000.0, 1151949900000.0, 1165945200000.0, 1177803400000.0, 1188600200000.0, 1198653700000.0, 1208132700000.0, 1217186100000.0, 1225809000000.0, 1234269200000.0, 1242689600000.0, 1251252200000.0, 1259951800000.0, 1268457100000.0, 1279594100000.0, 1288064300000.0, 1296732400000.0, 1306350800000.0, 1320749400000.0, 1368368600000.0, 1382908300000.0, 1394449200000.0, 1403701300000.0, 1421870400000.0, 1442400400000.0, 1459726700000.0, 1469102700000.0, 1477989500000.0, 1487951800000.0, 1513011500000.0, 1560199000000.0, 1570833800000.0, 1579861900000.0, 1589710300000.0, 1600094000000.0, 1609801100000.0, 1621373700000.0, 1629216000000.0, 1637721200000.0, 1656113300000.0, 1669808800000.0, 1678819000000.0, 1717836000000.0, 1776533500000.0, 1813434300000.0, 1823228400000.0, 1850019700000.0, 1866819000000.0, 1877642600000.0, 1930809000000.0, 1948741300000.0, 2024119300000.0, 2043187400000.0, 2065576000000.0, 2079388200000.0, 2136806900000.0, 2310955000000.0, 2327082700000.0, 2336245700000.0, 2360786300000.0, 2592580000000.0, 2962172000000.0, 3038592400000.0, 3052858500000.0, 3091299600000.0, 3104512700000.0, 3113781600000.0, 3118984300000.0, 3128386000000.0, 3134713000000.0, 3149529400000.0, 3163836000000.0, 3166528300000.0, 3167405700000.0]
    elif fofno == 2:
        Ndm_ar =  [506233, 543093, 582765, 625491, 671876, 722219, 777025, 836818, 901297, 970418, 1045227, 1126104, 1213949, 1309191, 1412584, 1524440, 1646551, 1777799, 1919728, 2072632, 2238248, 2417271, 2609800, 2817277, 3040510, 3281139, 3539926, 3817121, 4114546, 4432843, 4774753, 5139467, 5529198, 5945510, 6389364, 6861022, 7362273, 7896148, 8464347, 9073994, 9720932, 10401647, 11120919, 11885151, 12702569, 13567823, 14490623, 15490151, 16538923, 17659839, 18860064, 20139640, 21539654, 23004408, 24521251, 26118857, 27830425, 29690401, 31594495, 33583623, 35826373, 38084587, 40355477, 42603051, 44893225, 47240038, 49634641, 52131364, 54785123, 57566214, 60516069, 63630213, 66872280, 70032466, 73281595, 76559042, 79927775, 83271713, 86831349, 90335048, 93936908, 97529383, 101157411, 104732133, 108474590, 112510285, 116254382, 119352925, 122309945, 125330731, 128093855, 130971259, 133082588, 134592107, 135760830, 136423222, 136980286, 137712615, 138772681, 139954151]
        Mstar_ar = [622662400000.0, 639497900000.0, 656542900000.0, 673426200000.0, 690282760000.0, 707070850000.0, 723851700000.0, 741311400000.0, 758308100000.0, 774903000000.0, 791605700000.0, 808453400000.0, 825326700000.0, 842186600000.0, 858899000000.0, 875651200000.0, 892487200000.0, 909312900000.0, 925979840000.0, 942497300000.0, 959143700000.0, 975685350000.0, 992122900000.0, 1008428060000.0, 1024638000000.0, 1040602400000.0, 1056423740000.0, 1072317700000.0, 1088107640000.0, 1103894700000.0, 1119753900000.0, 1135668600000.0, 1151600600000.0, 1167671600000.0, 1184450500000.0, 1200219200000.0, 1215191000000.0, 1229984400000.0, 1244911900000.0, 1262749500000.0, 1279555500000.0, 1293580600000.0, 1306877000000.0, 1320081200000.0, 1332942500000.0, 1345256000000.0, 1357349500000.0, 1396579400000.0, 1409477200000.0, 1421382400000.0, 1434160100000.0, 1447370200000.0, 1486111800000.0, 1535126900000.0, 1546642500000.0, 1557432000000.0, 1571283800000.0, 1603248600000.0, 1616894800000.0, 1629796600000.0, 1693108300000.0, 1717380700000.0, 1780294600000.0, 1798316800000.0, 1806969000000.0, 1815239100000.0, 1824022600000.0, 1831566700000.0, 1840028700000.0, 1857419100000.0, 1873263900000.0, 1902653300000.0, 1939510400000.0, 1945850900000.0, 1952548900000.0, 1957783400000.0, 1971799000000.0, 1977263200000.0, 2014859800000.0, 2021033200000.0, 2033235000000.0, 2037815700000.0, 2059557500000.0, 2070316000000.0, 2093917300000.0, 2134067600000.0, 2198085600000.0, 2203074200000.0, 2214773400000.0, 2232973800000.0, 2254086000000.0, 2357980800000.0, 2370230600000.0, 2373513600000.0, 2384293500000.0, 2384670800000.0, 2385031500000.0, 2385912800000.0, 2392934600000.0, 2395122400000.0]


    rpl = np.logspace(1, 3.2, 100)
    rho_dm = Ndm_ar = np.array(Ndm_ar)*4.5e5 #This would be the density in Msun/kpc^3
        # Ndm_ar = Ndm_ar/Ndm_ar[-1]
    rho_star = Mstar_ar = np.array(Mstar_ar) #This would be the density in Msun/kpc^3
    rho_dm = Ndm_ar/((4/3.) * np.pi * rpl**3)
    rho_star = Mstar_ar/((4/3.) * np.pi * rpl**3)  
    Nstar_and_dm = (rho_dm + rho_star) / (rho_dm[-1] + rho_star[-1])

    Ndm_ar = Ndm_ar / (rho_dm[-1] + rho_star[-1]) /((4/3.) * np.pi * rpl**3)
    Mstar_ar = Mstar_ar / (rho_dm[-1] + rho_star[-1]) /((4/3.) * np.pi * rpl**3)

    
    # Nstar_ar = Nstar_ar/Nstar_ar[-1]
    



    pNm_ar = np.zeros(0) #shmf for merged subhalos
    pNm_ar_gp = np.zeros(0)
    pNs_ar = np.zeros(0) #shmf for surviving subhalos 
    pNs2_ar = np.zeros(0) #shmf for surviving subhalos 
    pNs2_wc_ar = np.zeros(0) #shmf for surviving subhalos 
    pNs_all_ar = np.zeros(0)
    
    pNs_ar1 = np.zeros(0) #shmf for surviving subhalos 
    pNs2_ar1 = np.zeros(0) #shmf for surviving subhalos 
    pNs2_wc_ar1 = np.zeros(0) #shmf for surviving subhalos 
    pNs_all_ar1 = np.zeros(0)

    
    pNs_ar2 = np.zeros(0) #shmf for surviving subhalos 
    pNs2_ar2 = np.zeros(0) #shmf for surviving subhalos 
    pNs2_wc_ar2 = np.zeros(0) #shmf for surviving subhalos 
    pNs_all_ar2 = np.zeros(0)
    
    cNm_ar = np.zeros(0) #shmf for merged subhalos
    cNs_ar = np.zeros(0) #shmf for surviving subhalos

    N_all_ar = np.zeros(0) #This is for all the subhalos inside virial radius
    Ntng_ar = np.zeros(0)
            
    mplcutoff = 0
    mmaxcutoff = 10**15
           
    rpl2 = np.logspace(1, np.log10(r200), 20)
    rpl21 = np.logspace(1, np.log10(r2001), 20)
    rpl22 = np.logspace(1, np.log10(r2002), 20)

        
        # fofno = 1

    for (ix, rs) in enumerate(rpl2): #rs is still radius, has nothing to do with rs of NFW profile
        pNm_ar = np.append(pNm_ar, len(pmmstar_f_ar[(pmmstar_all > mplcutoff) & (pmmstar_all < mmaxcutoff) & (pmvmx_if_ar > 45) & (pmdist_f_ar < rs) & (pmfof == fofno)]))
        pNm_ar_gp = np.append(pNm_ar_gp, len(pmmstar_f_ar[(pmmstar_all > mplcutoff) & (pmmstar_all < mmaxcutoff) & (pmvmx_if_ar > 45) & (pmgp_dist_f_ar < rs) & (pmfof == fofno)]))
        pNs_ar = np.append(pNs_ar, len(psmstar_f_ar[(psmstar_all > mplcutoff) & (psmstar_all < mmaxcutoff) & (psdist_f_ar < rs) & (psvmx_if_ar > 45) & (psfof == fofno) & (psmdm_f_ar/(4.5e5) > 100)]))
        pNs2_ar = np.append(pNs2_ar, len(psmstar_f_ar[(psmstar_all > mplcutoff) & (psmstar_all < mmaxcutoff) & (psdist_f_ar < rs) & (psvmx_f_ar > 30) & (psfof == fofno) & (psmdm_f_ar/(4.5e5) > 100)  ]))
        pNs2_wc_ar = np.append(pNs2_wc_ar, len(psmstar_f_ar[(psmstar_all > mplcutoff) & (psmstar_all < mmaxcutoff) & (psdist_f_ar < rs) & (psvmx_f_ar > 30) & (psfof == fofno) & (psmdm_f_ar/(4.5e5) > 100) & (psrapo_ar < r200) ]))
        pNs_all_ar = np.append(pNs_all_ar, len(psmstar_f_ar[(psmstar_all > mplcutoff) & (psmstar_all < mmaxcutoff) & (psdist_f_ar < rs) &  (psfof == fofno) & (psmdm_f_ar/(4.5e5) > 100) ]))
        
        pNs_ar1 = np.append(pNs_ar1, len(psmstar_f_ar[(psmstar_all > mplcutoff) & (psmstar_all < mmaxcutoff) & (psdist_f_ar < rpl21[ix]) & (psvmx_if_ar > 45) & (psfof == 1)& (psmdm_f_ar/(4.5e5) > 100)]))
        pNs2_ar1 = np.append(pNs2_ar1, len(psmstar_f_ar[(psmstar_all > mplcutoff) & (psmstar_all < mmaxcutoff) & (psdist_f_ar < rpl21[ix]) & (psvmx_f_ar > 30) & (psfof == 1) & (psmdm_f_ar/(4.5e5) > 100)]))
        pNs2_wc_ar1 = np.append(pNs2_wc_ar1, len(psmstar_f_ar[(psmstar_all > mplcutoff) & (psmstar_all < mmaxcutoff) & (psdist_f_ar < rpl21[ix]) & (psvmx_f_ar > 30) & (psfof == 1) & (psmdm_f_ar/(4.5e5) > 100)& (psrapo_ar < r2001)]))
        pNs_all_ar1 = np.append(pNs_all_ar1, len(psmstar_f_ar[(psmstar_all > mplcutoff) & (psmstar_all < mmaxcutoff) & (psdist_f_ar < rpl21[ix]) &  (psfof == 1) & (psmdm_f_ar/(4.5e5) > 100) ]))
        
        pNs_ar2 = np.append(pNs_ar2, len(psmstar_f_ar[(psmstar_all > mplcutoff) & (psmstar_all < mmaxcutoff) & (psdist_f_ar < rpl22[ix]) & (psvmx_if_ar > 45) & (psfof == 2)& (psmdm_f_ar/(4.5e5) > 100)]))
        pNs2_ar2 = np.append(pNs2_ar2, len(psmstar_f_ar[(psmstar_all > mplcutoff) & (psmstar_all < mmaxcutoff) & (psdist_f_ar < rpl22[ix]) & (psvmx_f_ar > 30) & (psfof == 2)& (psmdm_f_ar/(4.5e5) > 100)]))
        pNs2_wc_ar2 = np.append(pNs2_wc_ar2, len(psmstar_f_ar[(psmstar_all > mplcutoff) & (psmstar_all < mmaxcutoff) & (psdist_f_ar < rpl22[ix]) & (psvmx_f_ar > 30) & (psfof == 2)& (psmdm_f_ar/(4.5e5) > 100)& (psrapo_ar < r2002)]))
        pNs_all_ar2 = np.append(pNs_all_ar2, len(psmstar_f_ar[(psmstar_all > mplcutoff) & (psmstar_all < mmaxcutoff) & (psdist_f_ar < rpl22[ix]) &  (psfof == 2) & (psmdm_f_ar/(4.5e5) > 100) ]))
        # cNm_ar = np.append(cNm_ar, len(cmmstar_f_ar[(cmmstar_all > mplcutoff) & (cmmstar_all < mmaxcutoff) & (cmdist_f_ar < rs) & (cmfof == 0)]))
        # cNs_ar = np.append(cNs_ar, len(csmstar_f_ar[(csmstar_all > mplcutoff) & (csmstar_all < mmaxcutoff) & (csdist_f_ar < rs) & (csfof == 0)]))
        Ntng_ar = np.append(Ntng_ar, len(psmstar_f_ar_tng[(psmstar_f_ar_tng > mplcutoff) & (psmstar_f_ar_tng < mmaxcutoff) & (psvmx_if_ar > 45)  & (psdist_f_ar < rs) & (psfof == fofno)]))

        
        # axs[jx].text(0.1, 0.95, 'Nmerg/Ntot = ' + str(int(pNm_ar[-1])) + '/' + str(int(pNs_ar[-1] + pNm_ar[-1])), transform=axs[jx].transAxes, fontsize = 8)

    '''
    Following is the calculation of normalizing factors
    '''
    norm_Ns = ph_max_vmax_at_infall_n[24] * 0.4**3 * pNs_all_ar[-1] / len(psmstar_f_ar[(psmstar_all > mplcutoff) & (psmstar_all < mmaxcutoff) & (psdist_f_ar < 0.4 * r200) & (psvmx_if_ar > 45) & (psfof == fofno) & (psmdm_f_ar/(4.5e5) > 100)])  
    norm_Ns2 = ph_max_vmax_z0_n[28] * 0.4**3 * pNs_all_ar[-1] / len(psmstar_f_ar[(psmstar_all > mplcutoff) & (psmstar_all < mmaxcutoff) & (psdist_f_ar < 0.4 * r200) & (psvmx_f_ar > 30) & (psfof == fofno) & (psmdm_f_ar/(4.5e5) > 100)])
    norm_Ns2_wc = ph_max_vmax_z0_n[28] * 0.4**3 * pNs_all_ar[-1] / len(psmstar_f_ar[(psmstar_all > mplcutoff) & (psmstar_all < mmaxcutoff) & (psdist_f_ar < 0.4 * r200) & (psvmx_f_ar > 30) & (psfof == fofno) & (psmdm_f_ar/(4.5e5) > 100) & (psrapo_ar < r200) ]) 

    norm_Ns_1 = ph_max_vmax_at_infall_n[24] * 0.4**3 * pNs_all_ar1[-1] / len(psmstar_f_ar[(psmstar_all > mplcutoff) & (psmstar_all < mmaxcutoff) & (psdist_f_ar < 0.4 * r2001) & (psvmx_if_ar > 45) & (psfof == 1) & (psmdm_f_ar/(4.5e5) > 100)])  
    norm_Ns2_1 = ph_max_vmax_z0_n[28] * 0.4**3 * pNs_all_ar1[-1] / len(psmstar_f_ar[(psmstar_all > mplcutoff) & (psmstar_all < mmaxcutoff) & (psdist_f_ar < 0.4 * r2001) & (psvmx_f_ar > 30) & (psfof == 1) & (psmdm_f_ar/(4.5e5) > 100)])
    norm_Ns2_wc_1 = ph_max_vmax_z0_n[28] * 0.4**3 * pNs_all_ar1[-1] / len(psmstar_f_ar[(psmstar_all > mplcutoff) & (psmstar_all < mmaxcutoff) & (psdist_f_ar < 0.4 * r2001) & (psvmx_f_ar > 30) & (psfof == 1) & (psmdm_f_ar/(4.5e5) > 100) & (psrapo_ar < r2001) ]) 
    
    norm_Ns_2 = ph_max_vmax_at_infall_n[24] * 0.4**3 * pNs_all_ar2[-1] / len(psmstar_f_ar[(psmstar_all > mplcutoff) & (psmstar_all < mmaxcutoff) & (psdist_f_ar < 0.4 * r2002) & (psvmx_if_ar > 45) & (psfof == 2) & (psmdm_f_ar/(4.5e5) > 100)])  
    norm_Ns2_2 = ph_max_vmax_z0_n[28] * 0.4**3 * pNs_all_ar2[-1] / len(psmstar_f_ar[(psmstar_all > mplcutoff) & (psmstar_all < mmaxcutoff) & (psdist_f_ar < 0.4 * r2002) & (psvmx_f_ar > 30) & (psfof == 2) & (psmdm_f_ar/(4.5e5) > 100)])
    norm_Ns2_wc_2 = ph_max_vmax_z0_n[28] * 0.4**3 * pNs_all_ar2[-1] / len(psmstar_f_ar[(psmstar_all > mplcutoff) & (psmstar_all < mmaxcutoff) & (psdist_f_ar < 0.4 * r2002) & (psvmx_f_ar > 30) & (psfof == 2) & (psmdm_f_ar/(4.5e5) > 100) & (psrapo_ar < r2002) ]) 
    


    '''
    This part is for the calculation of numbers inside a given radius for the subhalos in the DMO simulation
    '''
    rpl2d = np.logspace(1, np.log10(fof0d_rvir), 20) # rpl2 for TNG50-1-Dark FoF0
    rpl21d = np.logspace(1, np.log10(fof1d_rvir), 20) # rpl2 for TNG50-1-Dark FoF1
    rpl22d = np.logspace(1, np.log10(fof2d_rvir), 20) # rpl2 for TNG50-1-Dark FoF2

    dNs2_ar = np.zeros(0)
    dNs_all_ar = np.zeros(0)

    d1Ns2_ar = np.zeros(0)
    d1Ns_all_ar = np.zeros(0)


    for ix in range(len(rpl2d)):
        dNs2_ar = np.append(dNs2_ar, len(len0d_ar[(vmax0d_ar > 30) & (dist0d_ar < rpl2d[ix]) & (len0d_ar > 100)]))
        dNs_all_ar = np.append(dNs_all_ar, len(len0d_ar[(dist0d_ar < rpl2d[ix]) & (len0d_ar > 100)]))

        d1Ns2_ar = np.append(d1Ns2_ar, len(len0d1_ar[(vmax0d1_ar > 30) & (dist0d1_ar < rpl21d[ix]) & (len0d1_ar > 100)]))
        d1Ns_all_ar = np.append(d1Ns_all_ar, len(len0d1_ar[(dist0d1_ar < rpl21d[ix]) & (len0d1_ar > 100)]))

    norm_dNs2 = ph_max_vmax_z0_n[28] * 0.4**3 * dNs_all_ar[-1] / len(len0d_ar[(vmax0d_ar > 30) & (dist0d_ar < 0.4 * fof0d_rvir) & (len0d_ar > 100)])


    norm_d1Ns2 = ph_max_vmax_z0_n[28] * 0.4**3 * d1Ns_all_ar[-1] / len(len0d1_ar[(vmax0d1_ar > 30) & (dist0d1_ar < 0.4 * fof0d1_rvir) & (len0d1_ar > 100)])

    '''
    This part is for the calculation of numberse inside a given radius for the subhalos in the TNG100-1-Dark simulation
    '''
    rpl2d1 = np.logspace(1, np.log10(fof0d1_rvir), 20) # rpl2 for TNG100-1-Dark FoF11
    rpl21d1 = np.logspace(1, np.log10(fof1d1_rvir), 20) # rpl2 for TNG100-1-Dark FoF12
    rpl22d1 = np.logspace(1, np.log10(fof2d1_rvir), 20) # rpl2 for TNG100-1-Dark FoF13

    rpl2m0d1 = np.logspace(1, np.log10(fofm0d1_rvir), 20) # rpl2 for TNG100-1-Dark FoF0
    # rpl2m1d1 = np.logspace(1, np.log10(fofm1d1_rvir), 20) # rpl2 for TNG100-1-Dark FoF12 

    d1Ns2_ar = np.zeros(0)
    d1Ns_all_ar = np.zeros(0)

    m0d1Ns2_ar = np.zeros(0)
    m0d1Ns_all_ar = np.zeros(0)

    for ix in range(len(rpl2d1)):
        d1Ns2_ar = np.append(d1Ns2_ar, len(len0d1_ar[(vmax0d1_ar > 30) & (dist0d1_ar < rpl2d1[ix]) & (len0d1_ar > 100)]))
        d1Ns_all_ar = np.append(d1Ns_all_ar, len(len0d1_ar[(dist0d1_ar < rpl2d1[ix]) & (len0d1_ar > 100)]))

        m0d1Ns2_ar = np.append(m0d1Ns2_ar, len(lenm0d1_ar[(vmaxm0d1_ar > 30) & (distm0d1_ar < rpl2m0d1[ix]) & (lenm0d1_ar > 100)]))
        m0d1Ns_all_ar = np.append(m0d1Ns_all_ar, len(lenm0d1_ar[(distm0d1_ar < rpl2m0d1[ix]) & (lenm0d1_ar > 100)]))

        

    norm_d1Ns2 = ph_max_vmax_z0_n[28] * 0.4**3 * d1Ns_all_ar[-1] / len(len0d1_ar[(vmax0d1_ar > 30) & (dist0d1_ar < 0.4 * fof0d1_rvir) & (len0d1_ar > 100)])
    norm_m0d1Ns2 = ph_max_vmax_z0_n[28] * 0.4**3 * m0d1Ns_all_ar[-1] / len(lenm0d1_ar[(vmaxm0d1_ar > 30) & (distm0d1_ar < 0.4 * fofm0d1_rvir) & (lenm0d1_ar > 100)])
                                
    

    # axs.plot(rpl2d/fof0d_rvir,  norm_Ns * pNs_ar/pNs_all_ar[-1] *  (rpl2[-1]**3 / rpl2**3), color = 'red', label = r'Power law - Type 1 > 45 km/s at inf, norm = '+str(int(norm_Ns)), lw  = 1.5, ls = '-')
    axs.plot(rpl2d/fof0d_rvir, norm_dNs2 *  dNs2_ar/dNs_all_ar[-1] *  (rpl2d[-1]**3 / rpl2d**3), color = 'magenta', label = r'Type 1 > 30 km/s at z = 0 (TNG50-Dark) FoF0, norm = '+str(int(norm_dNs2)), lw  = 1.5, ls = '-.')
    axs.plot(rpl2d1/fof0d1_rvir, norm_d1Ns2 *  d1Ns2_ar/d1Ns_all_ar[-1] *  (rpl2d1[-1]**3 / rpl2d1**3), color = 'violet', label = r'Type 1 > 30 km/s at z = 0 (TNG50-Dark) FoF1, norm = '+str(int(norm_d1Ns2)), lw  = 1.5, ls = '-.')
    
    axs.plot(rpl2d1/fof0d1_rvir, norm_d1Ns2 *  d1Ns2_ar/d1Ns_all_ar[-1] *  (rpl2d1[-1]**3 / rpl2d1**3), color = 'orange', label = r'Type 1 > 30 km/s at z = 0 (TNG100-Dark) FoF12, norm = '+str(int(norm_d1Ns2)), lw  = 1.5, ls = '-.')   
    axs.plot(rpl2m0d1/fofm0d1_rvir, norm_m0d1Ns2 *  m0d1Ns2_ar/m0d1Ns_all_ar[-1] *  (rpl2m0d1[-1]**3 / rpl2m0d1**3), color = 'brown', label = r'Type 1 > 30 km/s at z = 0 (TNG100-Dark) FoF0, norm = '+str(int(norm_m0d1Ns2)), lw  = 1.5, ls = '-.')
    
    
    Ntng_ar = Ntng_ar/Ntng_ar[-1] * (rpl2[-1]**3 / rpl2**3)
    pN_ar = (pNs_ar + pNm_ar) / (pNs_ar[-1] + pNm_ar[-1]) *  (rpl2[-1]**3 / rpl2**3)
    pN_ar_gp = (pNs_ar + pNm_ar_gp) / (pNs_ar[-1] + pNm_ar_gp[-1]) *  (rpl2[-1]**3 / rpl2**3)
    # cN_ar = (cNs_ar + cNm_ar) / (cNs_ar[-1] + cNm_ar[-1]) *  (rpl2[-1]**3 / rpl2**3)
    

    # axs[jx].plot(rpl2, Ntng_ar, 'bo-', label = r'TNG')
    # axs[jx].plot(rpl2, pN_ar, color = 'red', label = r'Power law', lw  = 1.5)
    axs.plot(rpl2/r200,  norm_Ns * pNs_ar/pNs_all_ar[-1] *  (rpl2[-1]**3 / rpl2**3), color = 'red', label = r'Power law - Type 1 > 45 km/s at inf, norm = '+str(int(norm_Ns)), lw  = 1.5, ls = '-')
    axs.plot(rpl2/r200,  norm_Ns2 * pNs2_ar/pNs_all_ar[-1] *  (rpl2[-1]**3 / rpl2**3), color = 'red', label = r'Power law - Type 1 > 30 km/s at z = 0, norm = '+str(int(norm_Ns)), lw  = 1.5, ls = '-.')
    # axs.plot(rpl2/r200,  norm_Ns2_wc * pNs2_wc_ar/pNs_all_ar[-1] *  (rpl2[-1]**3 / rpl2**3), color = 'red', label = r'Power law - Type 1 > 30 km/s at z = 0 - cut, norm = '+str(int(norm_Ns)), lw  = 1.5, ls = '-.')
    
    # axs.plot(rpl21/r2001,  norm_Ns_1 * pNs_ar1/pNs_all_ar1[-1] *  (rpl21[-1]**3 / rpl21**3), color = 'blue', label = r'Power law - Type 1 > 45 km/s at inf - FoF1, norm = '+str(int(norm_Ns_1)), lw  = 3, ls = '-')
    # axs.plot(rpl21/r2001,  norm_Ns2_1 * pNs2_ar1/pNs_all_ar1[-1] *  (rpl21[-1]**3 / rpl21**3), color = 'blue', label = r'Power law - Type 1 > 30 km/s at z = 0 - FoF1, norm = '+str(int(norm_Ns2_1)), lw  = 3, ls = '-.')
    # axs.plot(rpl21/r2001,  norm_Ns2_wc_1 * pNs2_wc_ar1/pNs_all_ar1[-1] *  (rpl21[-1]**3 / rpl21**3), color = 'blue', label = r'Power law - Type 1 > 30 km/s at z = 0 - FoF1, norm = '+str(int(norm_Ns2_wc_1)), lw  = 1.5, ls = '-.')

    
    # axs.plot(rpl22/r2002,  norm_Ns_2 * pNs_ar2/pNs_all_ar2[-1] *  (rpl22[-1]**3 / rpl22**3), color = 'green', label = r'Power law - Type 1 > 45 km/s at inf - FoF2, norm = '+str(int(norm_Ns_2)), lw  = 3, ls = '-')
    # axs.plot(rpl22/r2002,  norm_Ns2_2 * pNs2_ar2/pNs_all_ar2[-1] *  (rpl22[-1]**3 / rpl22**3), color = 'green', label = r'Power law - Type 1 > 30 km/s at z = 0 - FoF2, norm = '+str(int(norm_Ns2_2)), lw  = 3, ls = '-.')
    # axs.plot(rpl22/r2002,  norm_Ns2_wc_2 * pNs2_wc_ar2/pNs_all_ar2[-1] *  (rpl22[-1]**3 / rpl22**3), color = 'green', label = r'Power law - Type 1 > 30 km/s at z = 0 - cut - FoF2, norm = '+str(int(norm_Ns2_wc_2)), lw  = 1.5, ls = '-.')
    
    '''
    axs[jx].plot(rpl, Ndm_ar, color = 'black', ls = '--', label = 'DM in TNG', alpha = 0.5)
    axs[jx].plot(rpl, Nstar_and_dm, color = 'black', ls = '-', label = 'Stars and DM in TNG', alpha = 0.5)
    '''
    # axs[jx].plot(rpl, Mstar_ar, color = 'black', ls = ':', label = 'Stars in TNG', alpha = 0.5)
    # axs[jx].axvline(0.25 * r200, color = 'pink', ls = '--', label = r'$0.25 \times R_{\rm{200}}$')
    axs.plot(ph_dm_particles_r, ph_dm_particles_n, 'k--', label = 'DM - Phoenix', alpha = 0.5)
    axs.plot(ph_max_vmax_at_infall_r, ph_max_vmax_at_infall_n, 'k-', label = '> 45 km/s at inf - Phoenix', alpha = 0.5)
    axs.plot(ph_max_vmax_z0_r, ph_max_vmax_z0_n, 'k-.', label = '> 30 km/s at z = 0 - Phoenix', alpha = 0.5)

    
    axs.legend(fontsize = 6, loc = 'upper right')
    axs.set_ylabel(r'Number density (arbitrary normalization)')
    axs.set_xlabel(r'$r/r_{200}$')
    axs.set_ylim(bottom = 0.5, top = 1e2)

    
    axs.set_xscale('log')
    axs.set_yscale('log')
    axs.set_xlim(left = 0.05, right = 1)
    plt.tight_layout()
    # plt.savefig(this_fof_plotppath + 'radial_density_dist_3panel.png')
    plt.show()
    print(pNs_all_ar)
    print(rpl2[-2]/r200)

    return

plot_radial_density_dist_3panel(0)

